# Поиск объявлений для Авито

Нужно подобрать до 50 объявлений для каждого запроса. Качество будем оценивать по Recall@50: какую долю выбранных пользователями объявлений удалось найти.

Запускаем ячейки по порядку сверху вниз. Здесь загрузка данных, первый текстовый поиск TF-IDF, локальная проверка и сохранение ответа.

## 1. Подключаем библиотеку

`pandas` нужна для работы с таблицами. Называем её коротко `pd`, чтобы не писать полное имя каждый раз.

In [47]:
import pandas as pd

## 2. Читаем данные

Файлы лежат в папке `data` внутри проекта:
- `train` — примеры запросов и выбранных объявлений;
- `queries` — запросы, для которых нужно подготовить ответ;
- `items` — объявления, среди которых будем искать.

`read_parquet` читает файл и возвращает таблицу. Jupyter запускаем из папки проекта.

Для экономии памяти из train читаем только признаки запросов и идентификаторы выбранных объявлений. Тексты объявлений для поиска берём из items. Исходные файлы не изменяем.

In [48]:
search_columns = [
    "search_query",
    "search_location_id",
    "search_is_delivery_search",
    "search_infm_params_text",
    "search_category",
]

train = pd.read_parquet("data/train.parquet", columns=search_columns + ["item_id"])
queries = pd.read_parquet("data/benchmark_queries.parquet")
items = pd.read_parquet("data/benchmark_items.parquet")

## 3. Смотрим размер таблиц

`len` показывает число строк. В обучении одна строка — это пара «запрос и выбранное объявление».

In [49]:
print("Строк в обучении:", len(train))
print("Запросов для ответа:", len(queries))
print("Объявлений в корпусе:", len(items))

Строк в обучении: 497673
Запросов для ответа: 2452
Объявлений в корпусе: 189212


### Первые строки обучения

`head(3)` показывает первые три строки. Jupyter сам отображает таблицу, если она стоит последней строкой ячейки.

In [50]:
train.head(3)

,search_query,search_location_id,search_is_delivery_search,search_infm_params_text,search_category,item_id
0,скупка телевизоров,652430,0,,114,e8b685dffe1a408e
1,автоподбор,640860,0,Рейтинг пользователя 4 звезды и выше,114,92e1b0446f827b59
2,баня на дровах,653240,0,"Онлайн-запись Тип услуги СПА-услуги, массаж Ви...",114,624846856ce81d69


### Первые запросы для ответа

In [51]:
queries.head(3)

,query_id,search_query,search_location_id,search_is_delivery_search,search_infm_params_text,search_category
0,70DfDUpwjxB4lzFd,перевозки владикавказ тбилиси,649820,0,,114
1,JTrdTaZJvSiLPkXj,обзвон по базе,107620,0,Вид услуги Деловые услуги,114
2,LZCZNoVG4AFUkVRJ,липоредукция подбородка,637640,0,"Вид услуги Красота, здоровье",114


### Первые объявления корпуса

In [52]:
items.head(3)

,item_title_raw,item_rating_reviews_count,item_rating,item_price,item_microcat_id,item_longitude,item_location_id,item_latitude,item_is_phone_hidden,item_is_message_forbidden,item_infm_params_text,item_id,item_description_raw,item_category_id
0,Ремонт/выкуп компьют. и ноутбуков с выездом на...,57.0,5.0,500.000000000000000,2097573,42.047193961004901,631060,44.228180450924000,False,False,Вид услуги Компьютерная помощь Место оказания ...,111eb8b979577d79,"Выезд на дом в любое время, вплоть до 23:00\n\...",114
1,Обучение ребенка чтению,7.0,5.0,900.000000000000000,86453,49.627391820000000,631870,58.627609249999999,False,False,"Вид услуги Обучение, курсы Место оказания услу...",75fc8e10f5a66fc4,"Дорогие родители и маленькие книголюбы! \n\nЯ,...",114
2,Афрокудри 5+,14.0,5.0,1000.000000000000000,86467,40.936892000000000,628500,56.990307000000001,True,False,"Вид услуги Красота, здоровье Место оказания ус...",3f6ae81704565b9c,ВНИМАНИЕ Укладка !!!! 🥳\n\nАФРОкудри отличный ...,114


## 4. Смотрим колонки и пропуски

`info()` показывает названия колонок, их типы и количество заполненных значений. Идентификаторы оставляем строками: перевод в числа может испортить их.

In [53]:
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 497673 entries, 0 to 497672
Data columns (total 6 columns):
 #   Column                     Non-Null Count   Dtype
---  ------                     --------------   -----
 0   search_query               497673 non-null  str  
 1   search_location_id         497673 non-null  int64
 2   search_is_delivery_search  497673 non-null  int32
 3   search_infm_params_text    497673 non-null  str  
 4   search_category            497673 non-null  int64
 5   item_id                    497673 non-null  str  
dtypes: int32(1), int64(2), str(3)
memory usage: 73.6 MB


In [54]:
queries.info()

<class 'pandas.DataFrame'>
RangeIndex: 2452 entries, 0 to 2451
Data columns (total 6 columns):
 #   Column                     Non-Null Count  Dtype
---  ------                     --------------  -----
 0   query_id                   2452 non-null   str  
 1   search_query               2452 non-null   str  
 2   search_location_id         2452 non-null   int64
 3   search_is_delivery_search  2452 non-null   int32
 4   search_infm_params_text    2452 non-null   str  
 5   search_category            2452 non-null   int64
dtypes: int32(1), int64(2), str(3)
memory usage: 318.2 KB


In [55]:
items.info()

<class 'pandas.DataFrame'>
RangeIndex: 189212 entries, 0 to 189211
Data columns (total 14 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   item_title_raw             189212 non-null  str    
 1   item_rating_reviews_count  177597 non-null  float64
 2   item_rating                171581 non-null  float64
 3   item_price                 189212 non-null  object 
 4   item_microcat_id           189212 non-null  int64  
 5   item_longitude             189211 non-null  object 
 6   item_location_id           189212 non-null  int64  
 7   item_latitude              189211 non-null  object 
 8   item_is_phone_hidden       189212 non-null  bool   
 9   item_is_message_forbidden  189212 non-null  bool   
 10  item_infm_params_text      189212 non-null  str    
 11  item_id                    189212 non-null  str    
 12  item_description_raw       189177 non-null  str    
 13  item_category_id           189212 non-nu

`isna()` отмечает пустые значения. `sum()` считает их в каждой колонке. Пока только изучаем пропуски, ничего не удаляем.

In [56]:
train.isna().sum()

search_query                 0
search_location_id           0
search_is_delivery_search    0
search_infm_params_text      0
search_category              0
item_id                      0
dtype: int64

In [57]:
queries.isna().sum()

query_id                     0
search_query                 0
search_location_id           0
search_is_delivery_search    0
search_infm_params_text      0
search_category              0
dtype: int64

In [58]:
items.isna().sum()

item_title_raw                   0
item_rating_reviews_count    11615
item_rating                  17631
item_price                       0
item_microcat_id                 0
item_longitude                   1
item_location_id                 0
item_latitude                    1
item_is_phone_hidden             0
item_is_message_forbidden        0
item_infm_params_text            0
item_id                          0
item_description_raw            35
item_category_id                 0
dtype: int64

## 5. Проверяем идентификаторы

`assert` останавливает выполнение, если условие неверно, и показывает пояснение. Это поможет заметить проблему до формирования ответа.

У `query_id` должно быть 16 символов, у `item_id` — 16 символов из цифр и строчных букв a–f. Регулярное выражение `[0-9a-f]{16}` описывает этот формат.

In [59]:
assert queries["query_id"].notna().all(), "Есть пустые query_id"
assert queries["query_id"].is_unique, "Есть повторяющиеся query_id"
assert queries["query_id"].str.len().eq(16).all(), "Неверная длина query_id"

assert items["item_id"].notna().all(), "Есть пустые item_id"
assert items["item_id"].is_unique, "Есть повторяющиеся item_id"
assert items["item_id"].str.fullmatch("[0-9a-f]{16}", na=False).all(), "Неверный формат item_id"

print("Идентификаторы прошли проверку")

Идентификаторы прошли проверку


## 6. Посмотрим на запросы и объявления

Проверим, сколько разных текстов запроса есть в обучении и встречаются ли одинаковые условия поиска в запросах для ответа.
Список `search_columns` уже задан выше, при загрузке данных.

In [60]:
print("Уникальных текстов в обучении:", train["search_query"].nunique())
print("Повторов условий поиска:", queries.duplicated(search_columns).sum())

Уникальных текстов в обучении: 74529
Повторов условий поиска: 0


`isin` проверяет, встречается ли значение в другой таблице. Среднее `mean()` по значениям True и False даёт долю совпадений. Умножаем на 100, чтобы показать проценты.

Эти пересечения описывают данные, но не являются оценкой качества поиска.

In [61]:
known_queries = queries["search_query"].isin(train["search_query"])
known_items = train["item_id"].isin(items["item_id"])

print("Текст запроса был в обучении (%):", round(known_queries.mean() * 100, 2))
print("Выбранное объявление есть в корпусе (%):", round(known_items.mean() * 100, 2))

Текст запроса был в обучении (%): 36.99
Выбранное объявление есть в корпусе (%): 6.63


## 7. Подготавливаем текст

Сохраняем исходные колонки и создаём новые для поиска. Пустой текст заменяем пустой строкой, приводим буквы к нижнему регистру, заменяем ё на е, убираем знаки препинания и лишние пробелы.

Цифры оставляем: они могут быть частью модели техники. Слова пока не приводим к начальной форме. Цены, рейтинги и координаты не заполняем нулями: в этом базовом поиске они ещё не используются.

In [62]:
def clean_text(column):
    text = column.fillna("")
    text = text.str.lower()
    text = text.str.replace("ё", "е", regex=False)
    text = text.str.replace(r"[^а-яa-z0-9\s]", " ", regex=True)
    text = text.str.replace(r"\s+", " ", regex=True)
    return text.str.strip()

Для каждого объявления собираем один текст из заголовка, параметров и начала описания.
Заголовок повторяем два раза, чтобы его слова имели больший вес при поиске.
Описание ограничиваем 1500 символами, чтобы обработка не была слишком долгой.

In [63]:
titles = clean_text(items["item_title_raw"])
parameters = clean_text(items["item_infm_params_text"])
descriptions = clean_text(items["item_description_raw"].str.slice(0, 1500))

items["search_text"] = titles + " " + titles + " " + parameters + " " + descriptions

items[["item_title_raw", "search_text"]].head(3)

,item_title_raw,search_text
0,Ремонт/выкуп компьют. и ноутбуков с выездом на...,ремонт выкуп компьют и ноутбуков с выездом на ...
1,Обучение ребенка чтению,обучение ребенка чтению обучение ребенка чтени...
2,Афрокудри 5+,афрокудри 5 афрокудри 5 вид услуги красота здо...


## 8. Переводим тексты объявлений в числа

Используем TF-IDF из библиотеки scikit-learn. Он превращает каждый текст в набор чисел: слова, которые помогают отличить одно объявление от других, получают больший вес.

`fit_transform` сначала изучает слова в объявлениях, затем создаёт матрицу. В ней одна строка соответствует одному объявлению, а один столбец — одному слову.

In [64]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    min_df=2,
    max_features=100000,
    sublinear_tf=True,
    dtype=np.float32,
)

item_vectors = vectorizer.fit_transform(items["search_text"])
item_ids = items["item_id"].to_numpy()

print("Объявлений в индексе:", item_vectors.shape[0])
print("Слов в словаре:", item_vectors.shape[1])

Объявлений в индексе: 189212
Слов в словаре: 100000


## 9. Ищем объявления для одного запроса

`transform` переводит запрос в числа по тому же словарю, что и объявления.
Умножение `item_vectors @ query_vector.T` сравнивает каждый текст объявления с запросом. Чем больше полученное число, тем сильнее совпадение.

Затем берём до 50 объявлений с наибольшими оценками. Объявления с нулевой оценкой не добавляем: общих слов с запросом у них нет.

In [65]:
def find_candidates(query_text, filter_text="", top_k=50):
    query = clean_text(pd.Series([query_text])).iloc[0]

    # Если текст запроса пустой, пробуем искать по тексту фильтра.
    if query == "":
        query = clean_text(pd.Series([filter_text])).iloc[0]

    query_vector = vectorizer.transform([query])
    similarities = (item_vectors @ query_vector.T).toarray()[:, 0]

    # Сортируем оценки от большей к меньшей и оставляем первые позиции.
    best_positions = similarities.argsort()[::-1][:top_k]
    candidates = []

    for position in best_positions:
        if similarities[position] > 0:
            candidates.append(item_ids[position])

    return candidates

In [66]:
example_candidates = find_candidates("автоподбор")
items.set_index("item_id").loc[example_candidates[:10], ["item_title_raw", "item_location_id"]]

,item_title_raw,item_location_id
item_id,,
4ec3e697c2cfe3ee,"Автоподбор автомобиля, диагностика авто",654100
b09bb2c7397b4586,Автоподбор под ключ выездная диагностика,646600
a11f68d689210669,Автоподбор Автоэкспет,653401
ec47e0fdc1036a16,Автоподбор Санкт-Петербург,653240
f5883d50441eb38a,АВТОПОДБОР диагностика АВТО перед покупкой,648760
e9faeac4bd57c9ef,Осмотр и Автоподбор Без Хлопот,637640
78bebbf712a56a8f,Автоподбор,904751
a51fc121975fa918,Услуги автоподборщика/Эндоскопия двигателя,638720
338d395ae41fbab4,Автоподбор,661950


## 10. Создаём первую контрольную выборку

Объединяем строки train с одинаковыми пятью признаками запроса. Для каждого такого запроса собираем множество выбранных объявлений. Повторный выбор одного объявления не увеличивает его вес.

В корпусе нет большинства объявлений из train. Для этой проверки оставляем только группы, у которых **все известные положительные объявления** есть в корпусе. Затем случайно выбираем до 500 групп с фиксированным `random_state=42`.

Это ограниченная локальная проверка: распределение этих запросов может отличаться от бенчмарка, а неизвестные релевантные объявления нам не размечены. Пары train не используются для обучения TF-IDF. Когда начнём обучать модель на парах, потребуется отдельное разделение обучения и контроля.

In [67]:
validation_groups = train.groupby(search_columns, dropna=False)["item_id"].agg(set)
validation_groups = validation_groups.reset_index(name="relevant_items")

corpus_ids = set(items["item_id"])
can_evaluate = []

for relevant_items in validation_groups["relevant_items"]:
    can_evaluate.append(relevant_items.issubset(corpus_ids))

eligible_groups = validation_groups[can_evaluate]
validation = eligible_groups.sample(n=min(500, len(eligible_groups)), random_state=42)
validation = validation.reset_index(drop=True)

print("Всего групп запросов в train:", len(validation_groups))
print("Групп со всеми ответами в корпусе:", len(eligible_groups))
print("Запросов для локальной проверки:", len(validation))
assert len(validation) > 0, "Нет запросов для оценки в этом корпусе"

Всего групп запросов в train: 354463
Групп со всеми ответами в корпусе: 18436
Запросов для локальной проверки: 500


### Считаем Recall@50

Для каждого запроса пересекаем найденные объявления с известными выбранными. Число совпадений делим на число выбранных объявлений. Затем берём среднее по запросам.

In [68]:
recalls = []

for row_number, row in validation.iterrows():
    candidates = find_candidates(row["search_query"], row["search_infm_params_text"])
    relevant_items = row["relevant_items"]
    found_items = set(candidates) & relevant_items
    recalls.append(len(found_items) / len(relevant_items))

    if (row_number + 1) % 100 == 0:
        print("Проверено запросов:", row_number + 1)

validation["recall_at_50"] = recalls
baseline_recall = validation["recall_at_50"].mean()
print("Локальный Recall@50:", round(baseline_recall, 4))

Проверено запросов: 100
Проверено запросов: 200
Проверено запросов: 300
Проверено запросов: 400
Проверено запросов: 500
Локальный Recall@50: 0.2265


Посмотрим на запросы, где не удалось найти ни одного известного выбранного объявления. Они помогут понять, что улучшать: формы слов, синонимы, географию или обработку фильтров.

In [69]:
validation.loc[validation["recall_at_50"] == 0, search_columns].head(10)

,search_query,search_location_id,search_is_delivery_search,search_infm_params_text,search_category
0,нянч для ребенка,624600,0,"Вид услуги Няни, сиделки",114
1,грузоперевозки по россии дешево,636160,0,Вид услуги Грузоперевозки,114
2,установка дверных ручек,635430,0,Вид услуги Ремонт и отделка,114
3,натяжной потолок,624990,0,Тип услуги Потолки Вид услуги Ремонт и отделка,114
4,грузоперевозки по городу,637540,0,Вид услуги Грузоперевозки,114
5,ботокс для волос,653241,0,"Вид услуги Красота, здоровье",114
6,брови,646600,0,"Онлайн-запись Вид услуги Красота, здоровье",114
7,устранение засоров,652430,0,Тип услуги Сантехника Участие в пилоте CPL Вид...,114
8,обучение видеомонтажу,624770,0,,114
9,наращивание ногтей,638210,0,"Тип услуги Маникюр, педикюр Вид услуги Красота...",114


## 11. Получаем кандидатов для бенчмарка

Для всех запросов используем одну и ту же функцию поиска. `query_id` нужен только как ключ ответа и не участвует в вычислении сходства.

In [70]:
predictions = {}

for row_number, row in queries.iterrows():
    predictions[row["query_id"]] = find_candidates(
        row["search_query"], row["search_infm_params_text"]
    )

    if (row_number + 1) % 500 == 0:
        print("Обработано запросов:", row_number + 1)

print("Всего подготовлено ответов:", len(predictions))

Обработано запросов: 500
Обработано запросов: 1000
Обработано запросов: 1500
Обработано запросов: 2000
Всего подготовлено ответов: 2452


## 12. Проверка и сохранение готового ответа

`predictions` будет словарём: ключ — `query_id`, значение — список найденных `item_id`. `set` создаёт множество уникальных значений: так удобно проверить повторы и принадлежность корпусу.

Сначала проверяем все предсказания, затем сохраняем CSV. `join` соединяет идентификаторы одним пробелом. `index=False` запрещает добавлять лишнюю колонку с номерами строк.

In [71]:
def save_answer(predictions, queries, items, filename="answer.csv"):
    # В ответе должны быть все запросы и только они.
    expected_queries = set(queries["query_id"])
    if set(predictions) != expected_queries:
        raise ValueError("Список запросов не совпадает с benchmark")

    valid_items = set(items["item_id"])
    answers = []

    for query_id in queries["query_id"]:
        candidates = predictions[query_id]

        # Проверяем один список объявлений за раз.
        if not isinstance(candidates, list):
            raise TypeError("Кандидаты должны быть списком")

        if len(candidates) > 50:
            raise ValueError("Для одного запроса найдено больше 50 объявлений")

        for item_id in candidates:
            if not isinstance(item_id, str):
                raise TypeError("Идентификатор объявления должен быть строкой")
            if item_id not in valid_items:
                raise ValueError("Объявления нет в корпусе: " + item_id)

        if len(candidates) != len(set(candidates)):
            raise ValueError("В одном ответе повторяются объявления")

        answers.append(" ".join(candidates))

    answer = pd.DataFrame({
        "query_id": queries["query_id"].tolist(),
        "answer": answers,
    })

    answer.to_csv(filename, index=False, encoding="utf-8")

    # Читаем файл обратно и убеждаемся, что строки не изменились.
    saved_answer = pd.read_csv(filename, dtype=str, keep_default_na=False)
    pd.testing.assert_frame_equal(answer, saved_answer)

    print("Файл сохранён:", filename)
    return answer

Вызываем проверку и сохраняем первый ответ. Это базовое решение для сравнения с будущими улучшениями, а не финальная оптимизированная модель.

In [72]:
answer = save_answer(predictions, queries, items)
answer.head(3)

Файл сохранён: answer.csv


,query_id,answer
0,70DfDUpwjxB4lzFd,6042fecdd7753112 deb58d7ad8c27d06 603623b4bd9e...
1,JTrdTaZJvSiLPkXj,cbeccbecb1fb8d86 16ea26ff0aa0450a 8703b2a0442c...
2,LZCZNoVG4AFUkVRJ,3f89b8062dc85f1c d8fce513e4f000a7 c6e4d72fbddd...
